In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from pathlib import Path
import time
from tqdm import tqdm
import matplotlib.pyplot as plt

from Model import InpaintingDataset, UNetInpainting, loss_fn

In [2]:
# Configuration
IMAGE_SIZE = 256
BATCH_SIZE = 32
EPOCHS = 2000
TRAIN_SAMPLES_PER_EPOCH = 1024
VAL_SAMPLES_PER_EPOCH = 256
LR = 1e-4

CHECKPOINT_INTERVAL = 10
LOG_INTERVAL = 1
VAL_INTERVAL = 1

DATASET_PATH_TRAIN = "datasets/DIV2K/DIV2K_train_HR"
DATASET_PATH_TEST = "datasets/DIV2K/DIV2K_valid_HR"
CHECKPOINT_DIR = "checkpoints/inpainting"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Create checkpoint directory
Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)

Using device: cuda


In [3]:
# Initialize datasets
print("\n=== Creating Training Dataset ===")
train_dataset = InpaintingDataset(
    image_dir=DATASET_PATH_TRAIN,
    checkpoint_dir=CHECKPOINT_DIR,
    length=TRAIN_SAMPLES_PER_EPOCH,
    image_size=IMAGE_SIZE,
    num_strokes=(4, 5),
    thickness=(7, 9),
    mode='random',
)

print("\n=== Creating Validation Dataset ===")
val_dataset = InpaintingDataset(
    image_dir=DATASET_PATH_TEST,
    checkpoint_dir=CHECKPOINT_DIR,
    length=VAL_SAMPLES_PER_EPOCH,
    image_size=IMAGE_SIZE,
    num_strokes=(4, 5),
    thickness=(7, 9),
    mode='pregenerated',
)

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

# Initialize model
model = UNetInpainting().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

# Training state
start_epoch = 0
best_val_loss = float('inf')
train_losses = []
val_losses = []

# Load checkpoint if exists
checkpoint_path = Path(CHECKPOINT_DIR) / "latest_checkpoint.pth"
if checkpoint_path.exists():
    print(f"\n✓ Loading checkpoint from {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    best_val_loss = checkpoint.get('best_val_loss', float('inf'))
    train_losses = checkpoint.get('train_losses', [])
    val_losses = checkpoint.get('val_losses', [])
    print(f"  Resuming from epoch {start_epoch}")
    print(f"  Best validation loss: {best_val_loss:.6f}")
else:
    print("\n✗ No checkpoint found, starting from scratch")


=== Creating Training Dataset ===
✓ Random mode: picking from 800 images

=== Creating Validation Dataset ===
✗ Creating new dataset (pregenerated mode)
Pregenerating 256 samples...
  Generated 0/256
  Generated 100/256
  Generated 200/256
✓ Saved 256 samples to checkpoints\inpainting\pregenerated_data.pkl

✗ No checkpoint found, starting from scratch


In [ ]:
# Training loop
print(f"\n=== Starting Training ===")
print(f"Epochs: {EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Learning rate: {LR}")
print(f"Train samples per epoch: {TRAIN_SAMPLES_PER_EPOCH}")
print(f"Validation samples: {VAL_SAMPLES_PER_EPOCH}")

for epoch in range(start_epoch, EPOCHS):
    epoch_start_time = time.time()

    # ============ TRAINING ============
    model.train()
    train_loss = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{EPOCHS} [Train]", leave=False)
    for masked_img, mask, original_img in pbar:
        masked_img = masked_img.to(device)
        mask = mask.to(device)
        original_img = original_img.to(device)

        optimizer.zero_grad()

        # Forward pass
        predicted = model(masked_img, mask)

        # Calculate loss (only on holes)
        loss = loss_fn(predicted, original_img, mask)

        # Backward pass
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.6f}'})

    train_loss /= len(train_loader)
    train_losses.append(train_loss)

    # ============ VALIDATION ============
    if (epoch + 1) % VAL_INTERVAL == 0:
        model.eval()
        val_loss = 0

        with torch.no_grad():
            pbar = tqdm(val_loader, desc=f"Epoch {epoch + 1}/{EPOCHS} [Val]", leave=False)
            for masked_img, mask, original_img in pbar:
                masked_img = masked_img.to(device)
                mask = mask.to(device)
                original_img = original_img.to(device)

                # Forward pass
                predicted = model(masked_img, mask)

                # Calculate loss
                loss = loss_fn(predicted, original_img, mask)

                val_loss += loss.item()
                pbar.set_postfix({'loss': f'{loss.item():.6f}'})

        val_loss /= len(val_loader)
        val_losses.append(val_loss)

        # Check if best model
        is_best = val_loss < best_val_loss
        if is_best:
            best_val_loss = val_loss

        # ============ SAVE CHECKPOINT ============
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss,
            'best_val_loss': best_val_loss,
            'train_losses': train_losses,
            'val_losses': val_losses,
        }

        # Save latest checkpoint
        torch.save(checkpoint, checkpoint_path)

        # Save best model
        if is_best:
            best_path = Path(CHECKPOINT_DIR) / "best_model.pth"
            torch.save(checkpoint, best_path)

        # Save periodic checkpoint
        if (epoch + 1) % CHECKPOINT_INTERVAL == 0:
            periodic_path = Path(CHECKPOINT_DIR) / f"checkpoint_epoch_{epoch + 1}.pth"
            torch.save(checkpoint, periodic_path)

        # ============ VISUALIZE ============
        model.eval()
        fig, axes = plt.subplots(4, 4, figsize=(16, 16))

        with torch.no_grad():
            for i in range(4):
                masked_img, mask, original_img = val_dataset[i]

                masked_img_batch = masked_img.unsqueeze(0).to(device)
                mask_batch = mask.unsqueeze(0).to(device)

                predicted = model(masked_img_batch, mask_batch)

                masked_img = masked_img.cpu()
                mask = mask.cpu()
                original_img = original_img.cpu()
                predicted = predicted.squeeze(0).cpu()

                composite = masked_img * mask + predicted * (1 - mask)

                axes[i, 0].imshow(original_img.permute(1, 2, 0))
                axes[i, 0].set_title('Original')
                axes[i, 0].axis('off')

                axes[i, 1].imshow(masked_img.permute(1, 2, 0))
                axes[i, 1].set_title('Masked Input')
                axes[i, 1].axis('off')

                axes[i, 2].imshow(predicted.permute(1, 2, 0))
                axes[i, 2].set_title('Predicted')
                axes[i, 2].axis('off')

                axes[i, 3].imshow(composite.permute(1, 2, 0))
                axes[i, 3].set_title('Final Result')
                axes[i, 3].axis('off')

        plt.tight_layout()
        plt.savefig(Path(CHECKPOINT_DIR) / 'latest_results.png', dpi=150, bbox_inches='tight')
        plt.close()
    else:
        val_loss = None

    # ============ LOGGING ============
    if (epoch + 1) % LOG_INTERVAL == 0:
        epoch_time = time.time() - epoch_start_time

        log_msg = f"Epoch [{epoch + 1}/{EPOCHS}] "
        log_msg += f"Train Loss: {train_loss:.6f} "
        if val_loss is not None:
            log_msg += f"Val Loss: {val_loss:.6f} "
            if is_best:
                log_msg += "✓ NEW BEST "
        log_msg += f"Time: {epoch_time:.2f}s"

        print(log_msg)

print("\n=== Training Complete ===")
print(f"Best validation loss: {best_val_loss:.6f}")


=== Starting Training ===
Epochs: 1000
Batch size: 4
Learning rate: 0.0001
Train samples per epoch: 1024
Validation samples: 256


Epoch [1/1000] Train Loss: 0.386723 Val Loss: 0.260518 ✓ NEW BEST Time: 126.10s


Epoch [2/1000] Train Loss: 0.270261 Val Loss: 0.234667 ✓ NEW BEST Time: 112.86s


Epoch 3/1000 [Train]:  11%|█         | 28/256 [00:14<01:23,  2.75it/s, loss=0.257535]

In [ ]:
def plot_losses(train_losses, val_losses, val_interval):
    """
    Plot training and validation losses.

    Args:
        train_losses: List of training loss values (one per epoch)
        val_losses: List of validation loss values (one per val_interval epochs)
        val_interval: How often validation occurs (e.g., every 1, 5, 10 epochs)
    """
    # Create epoch arrays
    train_epochs = list(range(1, len(train_losses) + 1))
    val_epochs = list(range(val_interval, len(val_losses) * val_interval + 1, val_interval))

    # Plot
    plt.figure(figsize=(10, 6))
    plt.plot(train_epochs, train_losses, label='Train Loss', linewidth=2)
    plt.plot(val_epochs, val_losses, label='Val Loss', linewidth=2, marker='o', markersize=4)

    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_losses(train_losses, val_losses, VAL_INTERVAL)